# 🔍 Google News RSS - Player-Specific Feeds

## 🎯 Purpose

Ingest player-specific news from Google News RSS feeds into `main.fantasai.bronze_google_news`.

---

## 🔗 Data Source

**Source:** Google News RSS  
**Endpoint:** `https://news.google.com/rss/search?q={player_name}+NFL`  
**Cost:** ✅ FREE (no API key required)  
**Rate Limits:** ✅ None (RSS is public)  
**Coverage:** Aggregates 100+ news sources automatically  

---

## 📊 Benefits

* **Aggregation Power** - Google indexes hundreds of sources
* **Player-Specific** - Tailored feeds per player
* **Real-Time** - Near-instant updates
* **No Authentication** - Public RSS feeds
* **Free Forever** - No usage limits

---

## 📋 Data Fields

* `article_id` - Hash of (title + link)
* `player_id` - Master player ID
* `player_name` - Player name used in search
* `title` - Article title
* `description` - Article summary
* `link` - URL to article
* `published_at` - Publication timestamp
* `source` - Original publisher (extracted from title)
* `fetched_at` - Ingestion timestamp

---

## ⚙️ Execution

**Target Players:** Top 200 fantasy-relevant players (QB/RB/WR/TE)  
**Schedule:** ⏰ WEEKLY on Sundays at 7:00 AM UTC (after games complete)  
**Runtime:** ~2-3 minutes for 200 players  
**Dependencies:** Requires `gold_player_dim`

**Run Modes:**
* 📚 **HISTORICAL** - One-time setup load (fetch all current news, ~25K articles)
* 🔄 **INCREMENTAL** - Weekly updates (only articles from last 7 days, ~5-10K new articles/week)

**Incremental Strategy:**
* ✅ Fetch window: Only fetch articles published in last 7 days (incremental mode)
* 🗑️ Retention policy: Articles older than 60 days are auto-purged (rolling window)
* ✅ Deduplication via PRIMARY KEY (article_id, player_id)
* ✅ LEFT ANTI JOIN filters out existing articles before insert
* ✅ No duplicate articles ever written to bronze table
* ⚠️ **WARNING:** Do NOT run ad-hoc - use News Orchestrator job only  

---

## 📝 Notes

* Google News aggregates AP, ESPN, NFL.com, team sites, local beat writers
* RSS feeds return ~10 most recent articles per player
* Some articles mention multiple players (stored once per player)
* No player stats - purely news content

In [0]:
# =============================================================================
# GOOGLE NEWS RSS INGESTION - CONFIGURATION
# =============================================================================

from datetime import datetime
import time

print("="*80)
print("🔍 Google News RSS Ingestion Configuration")
print("="*80)

# === MODE SELECTION ===
RUN_MODE = "incremental"  # Options: "historical" (one-time load), "incremental" (weekly updates)
PLAYER_MODE = "production"  # Options: "test" (10 players), "production" (top 200)

# === RSS CONFIGURATION ===
GOOGLE_NEWS_RSS_URL = "https://news.google.com/rss/search"
REQUEST_TIMEOUT = 10  # seconds
RATE_LIMIT_DELAY = 0.05  # seconds between requests (fast, it's just RSS)

# === TABLE CONFIGURATION ===
BRONZE_TABLE = "main.fantasai.bronze_google_news"
PLAYER_DIM_TABLE = "main.fantasai.gold_player_dim"

# === PLAYER SELECTION ===
# Target top fantasy-relevant players (QB/RB/WR/TE)
TOP_N_PLAYERS = 200

# === DATE FILTERING (for incremental mode) ===
INCREMENTAL_DAYS_BACK = 7  # Only fetch articles from last N days in incremental mode
RETENTION_DAYS = 60  # Articles older than this are purged from the table (rolling window)

# === EXECUTION SETTINGS ===
if PLAYER_MODE == "test":
    PLAYER_LIMIT = 10
    print("\n🧪 TEST MODE - Processing 10 players")
else:
    PLAYER_LIMIT = TOP_N_PLAYERS
    print(f"\n⚡ PRODUCTION MODE - Processing top {TOP_N_PLAYERS} fantasy players")

if RUN_MODE == "historical":
    print(f"\n📚 HISTORICAL MODE - Loading all current news (establish baseline)")
    print("   ⚠️  This should only be run ONCE during initial setup")
else:
    print(f"\n🔄 INCREMENTAL MODE - Only articles from last {INCREMENTAL_DAYS_BACK} days")
    print("   📅 Designed for weekly runs (Sundays after games)")
    print(f"   🗑️  Articles older than {RETENTION_DAYS} days are purged (rolling window)")

print(f"\n📋 Configuration:")
print(f"   Run Mode: {RUN_MODE}")
print(f"   RSS Endpoint: {GOOGLE_NEWS_RSS_URL}")
print(f"   Bronze Table: {BRONZE_TABLE}")
print(f"   Source Table: {PLAYER_DIM_TABLE}")
print(f"   Rate Limit: {RATE_LIMIT_DELAY}s between requests")
print(f"   Request Timeout: {REQUEST_TIMEOUT}s")
print(f"   Player Limit: {PLAYER_LIMIT}")
if RUN_MODE == "incremental":
    print(f"   Fetch Window: Last {INCREMENTAL_DAYS_BACK} days")
    print(f"   Retention Window: {RETENTION_DAYS} days (auto-purge older articles)")

print("\n" + "="*80)

In [0]:
# =============================================================================
# PRE-FLIGHT CHECK - WARN IF RECENT RUN EXISTS
# =============================================================================

from datetime import datetime, timedelta

print("="*80)
print("🔍 Pre-Flight Safety Check")
print("="*80)

# Check for recent runs in the last 24 hours
try:
    recent_runs_df = spark.sql(f"""
        SELECT 
            MAX(fetched_at) as last_run,
            COUNT(DISTINCT DATE_TRUNC('hour', fetched_at)) as run_count_24h
        FROM {BRONZE_TABLE}
        WHERE fetched_at >= CURRENT_TIMESTAMP() - INTERVAL 24 HOURS
    """)
    
    row = recent_runs_df.first()
    
    if row and row['last_run']:
        last_run = row['last_run']
        run_count = row['run_count_24h']
        hours_since = (datetime.now() - last_run.replace(tzinfo=None)).total_seconds() / 3600
        
        print(f"\n🕔 Last Run: {last_run.strftime('%Y-%m-%d %H:%M:%S UTC')}")
        print(f"   Time since last run: {hours_since:.1f} hours ago")
        print(f"   Runs in last 24 hours: {run_count}")
        
        if hours_since < 1:
            print("\n⚠️⚠️⚠️ WARNING: A run completed less than 1 hour ago!")
            print("   Running again will likely find NO new articles (RSS feeds haven't refreshed)")
            print("   Recommended: Wait at least 6-12 hours between runs.\n")
        elif run_count > 1:
            print(f"\n⚠️  NOTICE: {run_count} runs detected in last 24 hours")
            print("   This notebook should only run weekly (Sundays) via orchestrator.\n")
        else:
            print("\n✅ Safe to proceed - no recent runs detected\n")
    else:
        print("\n✅ No recent runs found - safe to proceed\n")
        
except Exception as e:
    print(f"\n⚠️  Could not check recent runs (table may be empty): {e}")
    print("   Proceeding with caution...\n")

print("="*80)

In [0]:
# Install feedparser for RSS parsing
%pip install feedparser requests --quiet

print("✅ Dependencies installed")

In [0]:
%sql
-- Get top fantasy-relevant players for Google News RSS
-- Balanced sample: 50 players per position (QB, RB, WR, TE)

CREATE OR REPLACE TEMP VIEW top_fantasy_players AS
WITH ranked_players AS (
  SELECT 
    d.master_player_id,
    d.display_name as player_name,
    d.position,
    d.current_team as team,
    ROW_NUMBER() OVER (PARTITION BY d.position ORDER BY d.display_name) as rn
  FROM main.fantasai.gold_player_dim d
  WHERE d.position IN ('QB', 'RB', 'WR', 'TE')
    AND d.display_name IS NOT NULL
    AND d.display_name != ''
)
SELECT 
  master_player_id,
  player_name,
  position,
  team
FROM ranked_players
WHERE rn <= 50  -- Top 50 per position = 200 total
ORDER BY position, player_name;

-- Show sample
SELECT 
  COUNT(*) as total_players,
  COUNT(DISTINCT position) as positions,
  COUNT(DISTINCT team) as teams
FROM top_fantasy_players;

In [0]:
# =============================================================================
# FETCH GOOGLE NEWS RSS FEEDS FOR ALL PLAYERS
# =============================================================================

import feedparser
import requests
import hashlib
from datetime import datetime
import time
from typing import List, Dict, Optional
from urllib.parse import quote_plus

print("="*80)
print("📡 Fetching Google News RSS Feeds")
print("="*80)

# Get player list from temp view
players_df = spark.table("top_fantasy_players").toPandas()

if PLAYER_LIMIT:
    players_df = players_df.head(PLAYER_LIMIT)

print(f"\n📊 Processing {len(players_df)} players")
print(f"\n⏱️  Estimated time: ~{len(players_df) * RATE_LIMIT_DELAY / 60:.1f} minutes\n")

# === FETCH FUNCTION ===
def fetch_google_news_rss(player_name: str, timeout: int = REQUEST_TIMEOUT) -> Optional[Dict]:
    """Fetch Google News RSS feed for a player."""
    try:
        # Build search query: "Player Name" + NFL
        query = f"{player_name} NFL"
        encoded_query = quote_plus(query)
        url = f"{GOOGLE_NEWS_RSS_URL}?q={encoded_query}"
        
        # Parse RSS feed
        feed = feedparser.parse(url)
        
        if feed and hasattr(feed, 'entries'):
            return feed
        else:
            return None
    except Exception as e:
        return None

# === PROCESS ALL PLAYERS ===
all_articles = []
players_with_news = 0
total_articles = 0
errors = 0

for idx, row in players_df.iterrows():
    master_player_id = row['master_player_id']
    player_name = row['player_name']
    position = row['position']
    team = row['team']
    
    # Progress indicator every 50 players
    if (idx + 1) % 50 == 0:
        print(f"   ✓ Processed {idx + 1}/{len(players_df)} players ({players_with_news} with news, {total_articles} articles)")
    
    # Fetch RSS feed
    feed = fetch_google_news_rss(player_name)
    
    if feed and hasattr(feed, 'entries'):
        entries = feed.entries
        
        if entries:
            players_with_news += 1
            
            # Parse each article
            for entry in entries:
                try:
                    # Extract source from title (format: "Headline - Source Name")
                    title = entry.get('title', '')
                    source = 'Google News'
                    if ' - ' in title:
                        parts = title.rsplit(' - ', 1)
                        title = parts[0]
                        source = parts[1]
                    
                    # Get published date
                    published_at = entry.get('published', '')
                    if not published_at and hasattr(entry, 'published_parsed'):
                        published_at = datetime(*entry.published_parsed[:6]).isoformat() + 'Z'
                    
                    # Create unique article ID from title + link
                    link = entry.get('link', '')
                    article_id = hashlib.sha256(f"{title}{link}".encode()).hexdigest()[:16]
                    
                    # Build article record
                    article_record = {
                        'article_id': article_id,
                        'player_id': master_player_id,
                        'player_name': player_name,
                        'position': position,
                        'team': team,
                        'title': title,
                        'description': entry.get('summary', entry.get('description', '')),
                        'link': link,
                        'published_at': published_at,
                        'source': source,
                        'fetched_at': datetime.utcnow().isoformat() + 'Z'
                    }
                    
                    all_articles.append(article_record)
                    total_articles += 1
                    
                except Exception as e:
                    errors += 1
                    continue
    
    # Rate limiting (be polite)
    time.sleep(RATE_LIMIT_DELAY)

print("\n" + "="*80)
print("\n📊 Fetch Results:")
print(f"   Players Processed: {len(players_df)}")
print(f"   Players with News: {players_with_news} ({players_with_news/len(players_df)*100:.1f}%)")
print(f"   Total Articles: {total_articles}")
print(f"   Errors: {errors}")
if players_with_news > 0:
    print(f"   Avg Articles per Player (with news): {total_articles/players_with_news:.1f}")

if total_articles == 0:
    print("\n⚠️  No articles found. Check RSS endpoint or network.")
else:
    print(f"\n✅ Successfully fetched {total_articles} articles from Google News")

In [0]:
%sql
-- Create bronze table for Google News RSS articles (if not exists)

CREATE TABLE IF NOT EXISTS main.fantasai.bronze_google_news (
  article_id STRING NOT NULL COMMENT 'Hash of title + link',
  player_id STRING NOT NULL COMMENT 'Master player ID from gold_player_dim',
  player_name STRING COMMENT 'Player name used in search',
  position STRING COMMENT 'Player position',
  team STRING COMMENT 'Player team',
  title STRING COMMENT 'Article title',
  description STRING COMMENT 'Article summary/description',
  link STRING COMMENT 'URL to article',
  published_at TIMESTAMP COMMENT 'Publication timestamp',
  source STRING COMMENT 'Original publisher',
  fetched_at TIMESTAMP NOT NULL COMMENT 'Ingestion timestamp',
  CONSTRAINT pk_google_news PRIMARY KEY (article_id, player_id)
)
COMMENT 'Player-specific news from Google News RSS - Bronze layer'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

DESCRIBE EXTENDED main.fantasai.bronze_google_news;

In [0]:
# =============================================================================
# WRITE TO BRONZE TABLE WITH DEDUPLICATION
# =============================================================================

from pyspark.sql import functions as F

print("="*80)
print("💾 Writing Articles to Bronze Table")
print("="*80)

if total_articles == 0:
    print("\n⚠️  No articles to write. Skipping write operation.")
else:
    # Convert to Spark DataFrame
    articles_df = spark.createDataFrame(all_articles)
    
    # Apply incremental date filter BEFORE timestamp parsing (works on string timestamps)
    if RUN_MODE == "incremental":
        from datetime import datetime, timedelta
        cutoff_date = datetime.utcnow() - timedelta(days=INCREMENTAL_DAYS_BACK)
        cutoff_str = cutoff_date.isoformat() + 'Z'
        
        articles_before_filter = len(all_articles)
        all_articles_filtered = [a for a in all_articles if a.get('published_at', '') >= cutoff_str]
        articles_df = spark.createDataFrame(all_articles_filtered)
        
        print(f"\n📅 Incremental Mode: Filtered to articles from last {INCREMENTAL_DAYS_BACK} days (retention: {RETENTION_DAYS} days)")
        print(f"   Before filter: {articles_before_filter} articles")
        print(f"   After filter: {len(all_articles_filtered)} articles")
        print(f"   Filtered out: {articles_before_filter - len(all_articles_filtered)} older articles")
    
    # Convert timestamp strings to proper timestamps
    # RSS feeds use RFC 2822 format: "Thu, 04 Jun 2026 17:00:00 GMT"
    # Need to parse manually since Spark doesn't support this format natively
    from pyspark.sql.types import TimestampType
    from datetime import datetime as dt
    import email.utils
    
    # UDF to parse RFC 2822 timestamps
    def parse_rfc2822(ts_str):
        if not ts_str:
            return None
        try:
            # email.utils.parsedate_to_datetime handles RFC 2822 format
            return email.utils.parsedate_to_datetime(ts_str)
        except:
            # Fallback: try ISO format
            try:
                return dt.fromisoformat(ts_str.replace('Z', '+00:00'))
            except:
                return None
    
    parse_rfc2822_udf = F.udf(parse_rfc2822, TimestampType())
    
    articles_df = articles_df \
        .withColumn('published_at', parse_rfc2822_udf('published_at')) \
        .withColumn('fetched_at', F.to_timestamp('fetched_at'))
    
    print(f"\n📊 Prepared {articles_df.count()} articles for insertion")
    
    # Check for existing articles (deduplication)
    existing_articles_df = spark.sql(f"""
        SELECT DISTINCT article_id, player_id
        FROM {BRONZE_TABLE}
    """)
    
    existing_count = existing_articles_df.count()
    print(f"📋 Found {existing_count} existing articles in database")
    
    # Left anti join to find new articles only
    new_articles_df = articles_df.join(
        existing_articles_df,
        on=['article_id', 'player_id'],
        how='left_anti'
    )
    
    new_count = new_articles_df.count()
    duplicate_count = articles_df.count() - new_count
    
    print(f"\n🔍 Deduplication results:")
    print(f"   New articles: {new_count}")
    print(f"   Duplicates skipped: {duplicate_count}")
    
    if new_count > 0:
        print(f"\n💾 Writing {new_count} new articles to {BRONZE_TABLE}...")
        
        new_articles_df.write \
            .mode('append') \
            .saveAsTable(BRONZE_TABLE)
        
        print("\n✅ Write complete!")
        
        # Show sample of new articles
        print("\n🔍 Sample of new articles:")
        new_articles_df.select(
            'player_name',
            F.substring('title', 1, 60).alias('title_preview'),
            'source',
            'published_at'
        ).orderBy(F.desc('published_at')).show(10, truncate=False)
    else:
        print("\n✓ No new articles to write (all duplicates)")

print("\n" + "="*80)

In [0]:
%sql
-- =============================================================================
-- PURGE OLD ARTICLES - 60 DAY RETENTION WINDOW
-- =============================================================================
-- Remove articles older than 60 days to maintain a rolling window
-- This keeps storage manageable and focuses on recent, relevant news

DELETE FROM main.fantasai.bronze_google_news
WHERE published_at < CURRENT_TIMESTAMP() - INTERVAL 60 DAYS;

-- Show purge results
SELECT 
  'Purge Complete' as status,
  COUNT(*) as remaining_articles,
  MIN(published_at) as oldest_article,
  MAX(published_at) as newest_article,
  DATEDIFF(DAY, MIN(published_at), CURRENT_TIMESTAMP()) as oldest_article_age_days
FROM main.fantasai.bronze_google_news;

In [0]:
%sql
-- Summary statistics for Google News table

SELECT 
  COUNT(*) as total_articles,
  COUNT(DISTINCT player_id) as unique_players,
  COUNT(DISTINCT source) as unique_sources,
  MIN(published_at) as oldest_article,
  MAX(published_at) as newest_article,
  MAX(fetched_at) as last_ingestion_run,
  ROUND(AVG(LENGTH(title)), 0) as avg_title_length,
  ROUND(AVG(LENGTH(description)), 0) as avg_description_length
FROM main.fantasai.bronze_google_news;

In [0]:
%sql
-- Show ingestion history by run date
-- Confirms incremental behavior (no duplicates across runs)

WITH ingestion_runs AS (
  SELECT 
    DATE(fetched_at) as ingestion_date,
    COUNT(*) as articles_added,
    COUNT(DISTINCT player_id) as players_covered,
    COUNT(DISTINCT source) as unique_sources,
    MIN(published_at) as oldest_article,
    MAX(published_at) as newest_article
  FROM main.fantasai.bronze_google_news
  GROUP BY DATE(fetched_at)
  ORDER BY ingestion_date DESC
)
SELECT 
  ingestion_date,
  articles_added,
  players_covered,
  unique_sources,
  DATE_FORMAT(oldest_article, 'MMM dd, yyyy') as oldest_news,
  DATE_FORMAT(newest_article, 'MMM dd, yyyy') as newest_news
FROM ingestion_runs
LIMIT 10;